In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import csv
import time
import chardet

import geopandas as gpd

D:\seoulmate\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


In [2]:

# gis 데이터
geo_json = "행정동_emd_geojson.json"

area_file = "서울시_행정동_영역.csv"

#
# 법정동 위치크기 + 일단위 행정동 대중교통/버스/지하철 승차수
#
spatiotemporal_name = "서울시_행정동_시공간.csv"


In [3]:
# 1. geojson에서 행정동 영역 계산하기

gdf = gpd.read_file(geo_json)

gdf = gdf.to_crs(epsg=5181)  # 면적 계산용

gdf["AREA_M2"] = gdf.geometry.area

centroid = gdf.geometry.centroid

# 위경도로 변환
gdf_centroid = gpd.GeoSeries(centroid, crs="EPSG:5181").to_crs(epsg=4326)

gdf["LAT"] = gdf_centroid.x
gdf["LON"] = gdf_centroid.y

seoul = gdf[gdf["adm_cd2"].str.startswith("11")].copy()

print(f"행정동 갯수 : {len(seoul)}")



행정동 갯수 : 425


In [4]:
print(seoul.head(2))

   OBJECTID         adm_nm   adm_cd     adm_cd2    sgg sido sidonm sggnm  \
0         1  서울특별시 종로구 사직동  1101053  1111053000  11110   11  서울특별시   종로구   
1         2  서울특별시 종로구 삼청동  1101054  1111054000  11110   11  서울특별시   종로구   

                                            geometry       AREA_M2  \
0  MULTIPOLYGON (((197958.449 452900.684, 197971....  1.165780e+06   
1  MULTIPOLYGON (((198471.244 455055.39, 198531.5...  1.361445e+06   

          LAT        LON  
0  126.970144  37.574108  
1  126.981114  37.588013  


In [5]:

seoul["LAT"] = seoul["LAT"].round(5)
seoul["LON"] = seoul["LON"].round(5)

seoul["AREA_M2"] = seoul["AREA_M2"].round().astype(int)


# 필요한 컬럼만 선택
result = seoul[[
    "adm_cd2",
    "adm_nm",
    "AREA_M2",
    "LAT",
    "LON"
]].copy()

# CSV 저장
result.to_csv(
    area_file,
    index=False,
    encoding="utf-8-sig"
)

